In [1]:
#Lib Imports 
import pandas as pd
import numpy as np
import warnings

# ignorar todos los warnings
warnings.filterwarnings('ignore')


from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn_pandas import DataFrameMapper
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error


from tensorflow.keras.models import Sequential, clone_model,save_model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import Callback,EarlyStopping
from tensorflow.keras.optimizers import Adam



In [2]:

# cargamos el dataset principal con datos historicos y variables externas
df = pd.read_csv("../data/full_data.csv", parse_dates=["Date"], dayfirst=False)

# ordenamos por fecha
df = df.sort_values("Date").reset_index(drop=True)

# mostramos info general del dataset
print("columnas disponibles:")
print(df.columns.tolist())

print("\nprimeras filas:")
print(df.tail()) #Vemos los ultimos datos

print("\ntipos de datos:")
print(df.dtypes)


columnas disponibles:
['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Daily_Change', 'Volatility', 'Pct_Change', 'Volume_Change_pct', 'SMA_7', 'SMA_30', 'Rolling_volatility_30', 'BTC_Close_t-1', 'BTC_Close_t-2', 'BTC_Close_t-3', 'BTC_Close_t-7', 'fng_value', 'fng_classification', 'fng_diff_day', 'fng_SMA_7', 'fng_SMA_30', 'fng_trend', 'Is_Halving_Date', 'Block_reward']

primeras filas:
           Date         Close          High           Low          Open  \
3115 2026-08-13  63402.171875  63926.332031  62799.289062  63404.425781   
3116 2026-08-14  62975.593750  63551.562500  62487.699219  63402.089844   
3117 2026-08-15  63024.320312  63119.132812  62850.960938  62974.335938   
3118 2026-08-16  62818.652344  63310.777344  62648.574219  63023.421875   
3119 2026-08-18  64140.011719  64487.648438  64166.785156  64487.648438   

           Volume  Daily_Change   Volatility  Pct_Change  Volume_Change_pct  \
3115  18765583756     -2.253906  1127.042969   -0.000004          -0.186364  

In [3]:
# generacion de targets y analisis de correlacion

# generamos los 7 targets (precio de cierre futuro)
for i in range(1, 8):
    df[f"Close_t+{i}"] = df["Close"].shift(-i)

# eliminamos las filas sin datos completos (las ultimas 7)
df = df.dropna(subset=[f"Close_t+{i}" for i in range(1, 8)]).reset_index(drop=True)

# definimos las features numericas disponibles (todas menos las categoricas o de texto)
available_features = [col for col in df.columns if df[col].dtype != "object" and col not in [f"Close_t+{i}" for i in range(1, 8)]]

# lista de targets
targets = [f"Close_t+{i}" for i in range(1, 8)]


In [4]:
# seleccion de features mas correlacionadas

selected_features = ['Date','Volume','Pct_Change','Volume_Change_pct','Volatility','SMA_7','SMA_30',
             'BTC_Close_t-1','BTC_Close_t-2','BTC_Close_t-3','BTC_Close_t-7']


# dejamos solo las columnas seleccionadas y los 7 targets
keep_cols = selected_features + [f"Close_t+{i}" for i in range(1, 8)]
df = df[keep_cols].copy()

print(f"dataset final listo para entrenamiento, con {len(selected_features)} features y 7 targets\n")
print(df.head())


dataset final listo para entrenamiento, con 11 features y 7 targets

        Date       Volume  Pct_Change  Volume_Change_pct   Volatility  SMA_7  \
0 2018-02-01   9959400448         NaN                NaN  1476.519531    NaN   
1 2018-02-02  12726899712   -0.037052           0.277878  1345.790039    NaN   
2 2018-02-03   7263790080    0.038973          -0.429257  1179.120117    NaN   
3 2018-02-04   7073549824   -0.097865          -0.026190  1303.649902    NaN   
4 2018-02-05   9285289984   -0.159688           0.312678  1608.159668    NaN   

   SMA_30  BTC_Close_t-1  BTC_Close_t-2  BTC_Close_t-3  BTC_Close_t-7  \
0     NaN            NaN            NaN            NaN            NaN   
1     NaN    9170.540039            NaN            NaN            NaN   
2     NaN    8830.750000    9170.540039            NaN            NaN   
3     NaN    9174.910156    8830.750000    9170.540039            NaN   
4     NaN    8277.009766    9174.910156    8830.750000            NaN   

     Close_

In [5]:
# features: todas las columnas numericas excepto los targets y la fecha
targets = [f"Close_t+{i}" for i in range(1, 8)]
features = [c for c in df.columns if c not in targets + ["Date"]]

# diccionarios para guardar X e Y por horizonte
X_dict = {}
Y_dict = {}

for i in range(1, 8):
    tcol = f"Close_t+{i}"
    X_dict[i] = df[features].copy()
    Y_dict[i] = df[[tcol]].copy()

# mostrar shapes para confirmar
print("resumen de shapes por horizonte (i -> X.shape -> y.shape):\n")
for i in range(1, 8):
    print(f"t+{i}: X {X_dict[i].shape} -> Y {Y_dict[i].shape}")

# ejemplo: mostrar las primeras filas del horizonte 1
print("\nprimeras filas - ejemplo horizonte t+1 (X, Y):")
print(X_dict[1].head())
print(Y_dict[1].head())


resumen de shapes por horizonte (i -> X.shape -> y.shape):

t+1: X (3113, 10) -> Y (3113, 1)
t+2: X (3113, 10) -> Y (3113, 1)
t+3: X (3113, 10) -> Y (3113, 1)
t+4: X (3113, 10) -> Y (3113, 1)
t+5: X (3113, 10) -> Y (3113, 1)
t+6: X (3113, 10) -> Y (3113, 1)
t+7: X (3113, 10) -> Y (3113, 1)

primeras filas - ejemplo horizonte t+1 (X, Y):
        Volume  Pct_Change  Volume_Change_pct   Volatility  SMA_7  SMA_30  \
0   9959400448         NaN                NaN  1476.519531    NaN     NaN   
1  12726899712   -0.037052           0.277878  1345.790039    NaN     NaN   
2   7263790080    0.038973          -0.429257  1179.120117    NaN     NaN   
3   7073549824   -0.097865          -0.026190  1303.649902    NaN     NaN   
4   9285289984   -0.159688           0.312678  1608.159668    NaN     NaN   

   BTC_Close_t-1  BTC_Close_t-2  BTC_Close_t-3  BTC_Close_t-7  
0            NaN            NaN            NaN            NaN  
1    9170.540039            NaN            NaN            NaN  
2    8

In [6]:

#targets y features
targets = [f"Close_t+{i}" for i in range(1, 8)]  # 7 targets, 7 modelos
features = [col for col in df.columns if col not in targets + ['Date']]  # usamos todas las features excepto targets y date

#columnas a escalar
cols_scaler = features  # todas las numericas

#crear mapper
# mapper aplica: primero imputa valores nulos con la mediana, luego escala con robust scaler
mapper = DataFrameMapper([
    (cols_scaler, [SimpleImputer(strategy='median'), RobustScaler()])
], input_df=True, df_out=True)

#definir modelos
models_dict = {
    "LinearRegression": LinearRegression(),
    "KNN": KNeighborsRegressor(n_neighbors=15),
    "DecisionTree": DecisionTreeRegressor(max_depth=None, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42)
}

#pipelines vacios por cada target
pipelines = {}
for target in targets:
    # aca solo definimos el pipeline base, luego le metemos cada modelo de la lista, 28 pipelines
    pipelines[target] = {}
    
    for model_name, model in models_dict.items():
        pipeline = Pipeline([
            ('mapper', mapper),
            ('model', model)
        ])
        pipelines[target][model_name] = pipeline


In [7]:

# definimos tamaño de test final
test_size = 0.1
n_test = int(len(df) * test_size)

# separamos test final
train_val_df = df.iloc[:-n_test].reset_index(drop=True)
test_df = df.iloc[-n_test:].reset_index(drop=True)

n_splits = 7
tscv = TimeSeriesSplit(n_splits=n_splits)

# diccionarios para guardar índices por fold y por target
folds_idx = {target: [] for target in targets}

print(f"División en {n_splits} folds (train/val) por fechas:\n")
for target in targets:
    print(f" Target: {target}")
    X = train_val_df[features]
    y = train_val_df[[target]]

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        # guardar indices
        folds_idx[target].append((train_idx, val_idx))

        # fechas para mostrar
        start_train = train_val_df.iloc[train_idx[0]]['Date']
        end_train   = train_val_df.iloc[train_idx[-1]]['Date']
        start_val   = train_val_df.iloc[val_idx[0]]['Date']
        end_val     = train_val_df.iloc[val_idx[-1]]['Date']

        print(f"Fold {fold+1}:")
        print(f"  Train: {start_train.date()} -> {end_train.date()} ({len(train_idx)} filas)")
        print(f"  Val:   {start_val.date()} -> {end_val.date()} ({len(val_idx)} filas)\n")

# mostrar tamaño test final
print(f"Test final: {test_df['Date'].min().date()} -> {test_df['Date'].max().date()} ({len(test_df)} filas)")


División en 7 folds (train/val) por fechas:

 Target: Close_t+1
Fold 1:
  Train: 2018-02-01 -> 2019-01-18 (352 filas)
  Val:   2019-01-19 -> 2020-01-03 (350 filas)

Fold 2:
  Train: 2018-02-01 -> 2020-01-03 (702 filas)
  Val:   2020-01-04 -> 2020-12-18 (350 filas)

Fold 3:
  Train: 2018-02-01 -> 2020-12-18 (1052 filas)
  Val:   2020-12-19 -> 2021-12-03 (350 filas)

Fold 4:
  Train: 2018-02-01 -> 2021-12-03 (1402 filas)
  Val:   2021-12-04 -> 2022-11-18 (350 filas)

Fold 5:
  Train: 2018-02-01 -> 2022-11-18 (1752 filas)
  Val:   2022-11-19 -> 2023-11-03 (350 filas)

Fold 6:
  Train: 2018-02-01 -> 2023-11-03 (2102 filas)
  Val:   2023-11-04 -> 2024-10-18 (350 filas)

Fold 7:
  Train: 2018-02-01 -> 2024-10-18 (2452 filas)
  Val:   2024-10-19 -> 2025-10-03 (350 filas)

 Target: Close_t+2
Fold 1:
  Train: 2018-02-01 -> 2019-01-18 (352 filas)
  Val:   2019-01-19 -> 2020-01-03 (350 filas)

Fold 2:
  Train: 2018-02-01 -> 2020-01-03 (702 filas)
  Val:   2020-01-04 -> 2020-12-18 (350 filas)

Fol

In [8]:

n_splits = 7
tscv = TimeSeriesSplit(n_splits=n_splits)

# diccionario para resultados
cv_results = {}  # {target: {model_name: [fold_results]}}

for target in targets:
    print(f"\n Target: {target}")
    X = df[features]
    y = df[[target]]

    cv_results[target] = {}

    for model_name, pipeline in pipelines[target].items():
        print(f"\n Modelo: {model_name}")
        fold_results = []

        for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            # entrenar
            pipeline.fit(X_train, y_train)

            # predecir
            y_pred_train = pipeline.predict(X_train)
            y_pred_val   = pipeline.predict(X_val)

            # métricas
            mae_train = mean_absolute_error(y_train, y_pred_train)
            rmse_train = mean_squared_error(y_train, y_pred_train)**0.5

            mae_val = mean_absolute_error(y_val, y_pred_val)
            rmse_val = mean_squared_error(y_val, y_pred_val)**0.5

            fold_results.append({
                "fold": fold+1,
                "MAE_train": mae_train, "RMSE_train": rmse_train,
                "MAE_val": mae_val, "RMSE_val": rmse_val
            })

            print(f"Fold {fold+1}: Train MAE {mae_train:.2f}, RMSE {rmse_train:.2f} | "
                  f"Val MAE {mae_val:.2f}, RMSE {rmse_val:.2f}")

        cv_results[target][model_name] = fold_results



 Target: Close_t+1

 Modelo: LinearRegression
Fold 1: Train MAE 196.50, RMSE 298.50 | Val MAE 321.64, RMSE 445.16
Fold 2: Train MAE 210.90, RMSE 331.18 | Val MAE 808.58, RMSE 1464.32
Fold 3: Train MAE 411.85, RMSE 793.37 | Val MAE 1774.14, RMSE 2289.60
Fold 4: Train MAE 742.36, RMSE 1219.20 | Val MAE 483.14, RMSE 748.44
Fold 5: Train MAE 694.68, RMSE 1142.24 | Val MAE 896.50, RMSE 1423.22
Fold 6: Train MAE 740.10, RMSE 1212.20 | Val MAE 1895.01, RMSE 2533.04
Fold 7: Train MAE 919.26, RMSE 1486.07 | Val MAE 1670.10, RMSE 2276.68

 Modelo: KNN
Fold 1: Train MAE 315.58, RMSE 471.95 | Val MAE 980.69, RMSE 1201.03
Fold 2: Train MAE 345.94, RMSE 527.46 | Val MAE 11333.21, RMSE 19772.93
Fold 3: Train MAE 567.06, RMSE 1066.62 | Val MAE 3595.45, RMSE 4453.56
Fold 4: Train MAE 1557.41, RMSE 2324.76 | Val MAE 4824.48, RMSE 5440.78
Fold 5: Train MAE 1318.38, RMSE 1987.36 | Val MAE 3886.62, RMSE 5472.34
Fold 6: Train MAE 1568.33, RMSE 2328.49 | Val MAE 21095.89, RMSE 25935.11
Fold 7: Train MAE 201

In [9]:

# features y targets ya definidos

# diccionario para guardar pesos por fold/epoch
model_weights_by_horizon = {}

# Callback para guardar pesos
class OurCustomCallback(Callback):
    def __init__(self, horizon):
        super().__init__()
        self.horizon = horizon
        
    def on_epoch_end(self, epoch, logs=None):
        import copy
        if self.horizon not in model_weights_by_horizon:
            model_weights_by_horizon[self.horizon] = {}
        model_weights_by_horizon[self.horizon][epoch] = copy.deepcopy(self.model.get_weights())

# función para crear un modelo base
def create_mlp_model(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation='relu'),
        Dropout(0.1),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dense(1, activation='linear')  # salida para un solo target
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae','mse'])
    return model


In [10]:

# para guardar resultados
cv_results_nn = {}

# escalador
scaler = RobustScaler()

# early stopping para evitar sobreajuste
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

for i, target in enumerate(targets, start=1):
    print(f"\n Entrenando modelo NN para {target}")
    cv_results_nn[target] = []

    X = train_val_df[features].values
    y = train_val_df[[target]].values

    for fold, (train_idx, val_idx) in enumerate(folds_idx[target]):
        print(f"\n Fold {fold+1}/{len(folds_idx[target])}")

        # separar sets
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]


        # Se reemplazó por valores NaN
        imputer = SimpleImputer(strategy='median')
        X_train_imp = imputer.fit_transform(X_train)
        X_val_imp   = imputer.transform(X_val)
        X_train_scaled = scaler.fit_transform(X_train_imp)
        X_val_scaled   = scaler.transform(X_val_imp)


        # crear nuevo modelo
        model = create_mlp_model(X_train_scaled.shape[1])

        # callback personalizado (opcional)
        callback = OurCustomCallback(horizon=i)

        # entrenar
        history = model.fit(
            X_train_scaled, y_train,
            validation_data=(X_val_scaled, y_val),
            epochs=150,
            batch_size=32,
            verbose=0
        )

        # predicciones
        y_pred_train = model.predict(X_train_scaled)
        y_pred_val = model.predict(X_val_scaled)

        # métricas
        mae_train = mean_absolute_error(y_train, y_pred_train)
        rmse_train = mean_squared_error(y_train, y_pred_train) ** 0.5
        mape_train = mean_absolute_percentage_error(y_train, y_pred_train)

        mae_val = mean_absolute_error(y_val, y_pred_val)
        rmse_val = mean_squared_error(y_val, y_pred_val) ** 0.5
        mape_val = mean_absolute_percentage_error(y_val, y_pred_val)

        print(
            f"Fold {fold+1}: "
            f"Train -> MAE={mae_train:.4f}, RMSE={rmse_train:.4f} | "
            f"Val -> MAE={mae_val:.4f}, RMSE={rmse_val:.4f}"
        )

        # guardar resultados
        cv_results_nn[target].append({
            'fold': fold+1,
            'MAE_train': mae_train,
            'RMSE_train': rmse_train,
            'MAPE_train': mape_train,
            'MAE_val': mae_val,
            'RMSE_val': rmse_val,
            'MAPE_val': mape_val
        })

# promedio final por target
print("\n Resultados Promedio por Horizonte")
for target in targets:
    mae_train_mean = np.mean([r['MAE_train'] for r in cv_results_nn[target]])
    mae_val_mean = np.mean([r['MAE_val'] for r in cv_results_nn[target]])
    rmse_train_mean = np.mean([r['RMSE_train'] for r in cv_results_nn[target]])
    rmse_val_mean = np.mean([r['RMSE_val'] for r in cv_results_nn[target]])

    print(
        f"{target}: "
        f"Train -> MAE={mae_train_mean:.4f}, RMSE={rmse_train_mean:.4f} | "
        f"Val -> MAE={mae_val_mean:.4f}, RMSE={rmse_val_mean:.4f}"
    )



 Entrenando modelo NN para Close_t+1

 Fold 1/7
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
Fold 1: Train -> MAE=235.5609, RMSE=328.6812 | Val -> MAE=2586.0130, RMSE=3464.7875

 Fold 2/7
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Fold 2: Train -> MAE=206.6055, RMSE=308.2418 | Val -> MAE=455.1612, RMSE=692.3865

 Fold 3/7
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Fold 3: Train -> MAE=214.4260, RMSE=333.8807 | Val -> MAE=2750.7612, RMSE=3561.5915

 Fold 4/7
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Fold 4: Train -> MAE=606.6442, RMSE=1104.0050 | Val -> MAE=1037.7067, RMSE=1370.9845

 Fold 5/7
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Fold 5: Train -> MAE=672.1836, RMSE=1154.1436 | Val -> MAE=481.6309, RMSE=680.0220

 Fold 6/7
66/66 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
F

In [11]:
import os
import joblib  # para guardar el scaler

# directorio para guardar modelos
output_dir = "./models_final"
os.makedirs(output_dir, exist_ok=True)

# diccionario para guardar modelos y resultados
final_models = {}
final_metrics = {}

# iterar sobre los 7 horizontes
for i, target in enumerate(targets, start=1):
    print(f"\n Entrenando modelo final para {target}")
    
    # X e y completos (train + val)
    X = train_val_df[features].values
    y = train_val_df[[target]].values

    # imputar NaN antes de escalar -- Evitar error NaN
    imputer_final = SimpleImputer(strategy='median')
    X_imp = imputer_final.fit_transform(X)
    # escalar
    scaler_final = RobustScaler()
    X_scaled = scaler_final.fit_transform(X_imp)
    
    # crear modelo
    model = create_mlp_model(X_scaled.shape[1])

    # entrenar
    history = model.fit(
        X_scaled, y,
        epochs=200,
        batch_size=32,
        verbose=0
    )

    # predicciones sobre todo el dataset
    y_pred = model.predict(X_scaled)

    # calcular metricas
    mae = mean_absolute_error(y, y_pred)
    rmse = mean_squared_error(y, y_pred) ** 0.5
    mape = mean_absolute_percentage_error(y, y_pred)

    print(f"{target} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

    model.save(os.path.join(output_dir, f"mlp_{target}.keras"))
    joblib.dump(scaler_final,  os.path.join(output_dir, f"scaler_{target}.pkl"))
    joblib.dump(imputer_final, os.path.join(output_dir, f"imputer_{target}.pkl"))
    print(f"artefactos guardados en {output_dir}")

    # guardar en diccionario para uso inmediato
    final_models[target] = {
        "model": model,
        "scaler": scaler_final
    }

    # guardar metricas
    final_metrics[target] = {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    }

print("\n Todos los modelos finales, scalers y metricas guardados")
final_metrics[target] = {"MAE": mae, "RMSE": rmse, "MAPE": mape}


 Entrenando modelo final para Close_t+1
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Close_t+1 -> MAE: 825.8276, RMSE: 1388.0985, MAPE: 0.0277
artefactos guardados en ./models_final

 Entrenando modelo final para Close_t+2
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Close_t+2 -> MAE: 1243.2799, RMSE: 2019.1451, MAPE: 0.0389
artefactos guardados en ./models_final

 Entrenando modelo final para Close_t+3
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Close_t+3 -> MAE: 1402.5928, RMSE: 2220.2475, MAPE: 0.0476
artefactos guardados en ./models_final

 Entrenando modelo final para Close_t+4
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Close_t+4 -> MAE: 1595.4054, RMSE: 2526.9812, MAPE: 0.0527
artefactos guardados en ./models_final

 Entrenando modelo final para Close_t+5
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Close_t+5 -> MAE: 1783.9573, RMSE: 2820.1917, MAPE: 0.0581
artefactos guardados en ./models_final

 Entrenando modelo final para Close_t+6
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Close_t+6 -> MAE: 1980.3967, RM

In [ ]:


# diccionario para guardar métricas de test
test_metrics = {}

for target in targets:
    print(f"\n Evaluando modelo en test para {target}")

    # obtener modelo y scaler desde final_models
    model = final_models[target]["model"]
    scaler = final_models[target]["scaler"]

    # preparar X e y de test
    X_test = test_df[features].values
    y_test = test_df[[target]].values
        
     # aplicar los mismos artefactos del train, sin refitear
    X_test_imp    = imputer_final.transform(X_test)
    X_test_scaled = scaler.transform(X_test_imp)

    # predecir
    y_pred_test = model.predict(X_test_scaled)

    # calcular métricas
    mae = mean_absolute_error(y_test, y_pred_test)
    rmse = mean_squared_error(y_test, y_pred_test) ** 0.5
    mape = mean_absolute_percentage_error(y_test, y_pred_test)

    print(f"{target} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

    # guardar métricas
    test_metrics[target] = {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    }

print("\n métricas finales sobre test")
for target, met in test_metrics.items():
    print(f"{target}: MAE={met['MAE']:.4f}, RMSE={met['RMSE']:.4f}, MAPE={met['MAPE']:.4f}")



 Evaluando modelo en test para Close_t+1
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
Close_t+1 -> MAE: 1677.9428, RMSE: 2320.0984, MAPE: 0.0208

 Evaluando modelo en test para Close_t+2
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Close_t+2 -> MAE: 2246.1609, RMSE: 3081.0596, MAPE: 0.0279

 Evaluando modelo en test para Close_t+3
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Close_t+3 -> MAE: 2514.0836, RMSE: 3343.0596, MAPE: 0.0310

 Evaluando modelo en test para Close_t+4
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Close_t+4 -> MAE: 2951.9682, RMSE: 4046.9369, MAPE: 0.0365

 Evaluando modelo en test para Close_t+5
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Close_t+5 -> MAE: 3146.1589, RMSE: 4261.4779, MAPE: 0.0388

 Evaluando modelo en test para Close_t+6
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Close_t+6 -> MAE: 3549.5183, RMSE: 4861.8477, MAPE: 0.0441

 Evaluando modelo en test para Close_t+7
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Close_t+7 -> MAE: 4156.9106, RMSE: 5741.9662, MAPE: 0.0520

 métricas fi

: 